# Explainable Customer Segmentation

Research experiments for RFM clustering, adaptive-recency comparison and FCA-style explanations. Install `requirements.txt` first; set the data path in the configuration below. Outputs were cleared during cleanup and must be regenerated with your data. Historical results are documented in the README. Prediction training is maintained separately and has not yet been uploaded.

Run cells in order. The original feature engineering that produces the input CSV is not included. Full-data experiments and silhouette sampling can require substantial memory and time.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

# Override these environment variables before starting the notebook if needed.
INPUT_CSV = Path(os.environ.get("CUSTOMER_DATA_CSV", "data/customer_features.csv"))
OUTPUT_DIR = Path(os.environ.get("CUSTOMER_OUTPUT_DIR", "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not INPUT_CSV.is_file():
    raise FileNotFoundError(
        f"Customer feature file not found: {INPUT_CSV}. "
        "See data/README.md for the required schema and configuration."
    )
required_columns = {"customer_id", "recency", "frequency", "monetary", "adaptive_recency"}
input_columns = set(pd.read_csv(INPUT_CSV, nrows=0).columns)
if required_columns - input_columns:
    raise ValueError(f"Missing columns: {sorted(required_columns - input_columns)}")


In [ ]:
import pandas as pd

# load file
df = pd.read_csv(INPUT_CSV)

# check column names
print(df.columns)

# if you want them in list form
print(df.columns.tolist())

In [ ]:
import pandas as pd

# load file
df = pd.read_csv(INPUT_CSV)

# keep only the columns you want
selected_columns = [
    'customer_id',
    'recency',
    'adaptive_recency',
    'frequency',
    'monetary'
]

# create new dataframe
df_selected = df[selected_columns]

# save new file
df_selected.to_csv(str(OUTPUT_DIR / "final_selected_features.csv"), index=False)

print(df_selected.head())
print("New file saved as final_selected_features.csv")

## HDBSCAN on RFM

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(str(OUTPUT_DIR / "final_selected_features.csv"))

features = ['recency', 'adaptive_recency', 'frequency', 'monetary']

df = df[['customer_id'] + features].copy()
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

print(df.shape)
print(df.head())

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import pandas as pd
import numpy as np

df = pd.read_csv(INPUT_CSV)

features = ['recency',  'frequency', 'monetary']

df = df[['customer_id'] + features].copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna()

scaler = StandardScaler()
X_full_scaled = scaler.fit_transform(df[features])

print("Full data scaled:", X_full_scaled.shape)

In [ ]:
features = ['recency', 'frequency', 'monetary']

X = df[features].astype('float32')

print(X.dtypes)

In [ ]:
features = ['recency', 'frequency', 'monetary']

X = df[features].astype('float32')

print(X.head())

In [ ]:
from sklearn.preprocessing import StandardScaler
import gc

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X).astype('float32')

del X
gc.collect()

print(X_scaled.shape)

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15000,
    min_samples=30,
    metric='euclidean',
    core_dist_n_jobs=1
)

labels = clusterer.fit_predict(X_scaled)

df['hdbscan_cluster'] = labels

In [ ]:
print(df['hdbscan_cluster'].value_counts().sort_index())

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_pct = (labels == -1).sum() / len(labels) * 100

print("Clusters:", n_clusters)
print("Noise %:", round(noise_pct, 2))

In [ ]:
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

# remove noise points
mask = labels != -1

X_valid = X_scaled[mask]
labels_valid = labels[mask]

print("Valid points:", len(labels_valid))
print("Clusters:", len(set(labels_valid)))

# Evaluation
sil = silhouette_score(
    X_valid,
    labels_valid,
    sample_size=min(20000, len(labels_valid)),
    random_state=42
)

ch = calinski_harabasz_score(X_valid, labels_valid)

db = davies_bouldin_score(X_valid, labels_valid)

print("\n===== HDBSCAN Evaluation =====")
print("Silhouette Score:", sil)
print("Calinski-Harabasz Score:", ch)
print("Davies-Bouldin Score:", db)

## HDBSCAN on RFM + adaptive recency

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import pandas as pd
import numpy as np

df = pd.read_csv(INPUT_CSV)

features = ['recency', 'adaptive_recency' ,'frequency', 'monetary']

df = df[['customer_id'] + features].copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna()

scaler = StandardScaler()
X_full_scaled = scaler.fit_transform(df[features])

print("Full data scaled:", X_full_scaled.shape)

In [ ]:
features = ['recency', 'frequency','adaptive_recency', 'monetary']

X = df[features].astype('float32')

print(X.dtypes)

In [ ]:
from sklearn.preprocessing import StandardScaler
import gc

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X).astype('float32')

del X
gc.collect()

print(X_scaled.shape)

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15000,
    min_samples=30,
    metric='euclidean',
    core_dist_n_jobs=1
)

labels = clusterer.fit_predict(X_scaled)

df['hdbscan_cluster'] = labels

In [ ]:
print(df['hdbscan_cluster'].value_counts().sort_index())

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_pct = (labels == -1).sum() / len(labels) * 100

print("Clusters:", n_clusters)
print("Noise %:", round(noise_pct, 2))

In [ ]:
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

# remove noise points
mask = labels != -1

X_valid = X_scaled[mask]
labels_valid = labels[mask]

print("Valid points:", len(labels_valid))
print("Clusters:", len(set(labels_valid)))

# Evaluation
sil = silhouette_score(
    X_valid,
    labels_valid,
    sample_size=min(20000, len(labels_valid)),
    random_state=42
)

ch = calinski_harabasz_score(X_valid, labels_valid)

db = davies_bouldin_score(X_valid, labels_valid)

print("\n===== HDBSCAN Evaluation =====")
print("Silhouette Score:", sil)
print("Calinski-Harabasz Score:", ch)
print("Davies-Bouldin Score:", db)

In [ ]:
import pandas as pd
import numpy as np
import gc

df = pd.read_csv(
    str(OUTPUT_DIR / "final_selected_features.csv"),
    usecols=['customer_id', 'recency', 'frequency', 'monetary']
)

df = df.replace([np.inf, -np.inf], np.nan).dropna()

print(df.shape)

In [ ]:
import pandas as pd

# load file
df = pd.read_csv(str(OUTPUT_DIR / "final_selected_features.csv"))

# correlation between recency and adaptive recency
correlation = df['recency'].corr(df['adaptive_recency'])

print("Correlation between Recency and Adaptive Recency:")
print(correlation)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr_matrix = df[['recency', 'adaptive_recency']].corr()

plt.figure(figsize=(5,4))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')

plt.title('Correlation Between Recency and Adaptive Recency')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "heatmap.png"))
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# load file
df = pd.read_csv(str(OUTPUT_DIR / "final_selected_features.csv"))

# ---------------- RECENCY HISTOGRAM ----------------
plt.figure(figsize=(7,4))
plt.hist(df['recency'], bins=30)
plt.xlabel('Recency')
plt.ylabel('Number of Customers')
plt.title('Distribution of Recency')
plt.show()

# ---------------- FREQUENCY HISTOGRAM ----------------
plt.figure(figsize=(7,4))
plt.hist(df['frequency'], bins=30)
plt.xlabel('Frequency')
plt.ylabel('Number of Customers')
plt.title('Distribution of Frequency')
plt.show()

# ---------------- MONETARY HISTOGRAM ----------------
plt.figure(figsize=(7,4))
plt.hist(df['monetary'], bins=30)
plt.xlabel('Monetary')
plt.ylabel('Number of Customers')
plt.title('Distribution of Monetary')
plt.show()

In [ ]:
# RECENCY
plt.figure(figsize=(6,4))
plt.hist(df['recency'], bins=30)
plt.xlabel('Recency')
plt.ylabel('Number of Customers')
plt.title('Distribution of Recency')

plt.savefig(str(OUTPUT_DIR / "recency_histogram.png"))
plt.show()

In [ ]:
# FREQUENCY
plt.figure(figsize=(6,4))
plt.hist(df['frequency'], bins=30)
plt.xlabel('Frequency')
plt.ylabel('Number of Customers')
plt.title('Distribution of Frequency')
plt.tight_layout()


plt.savefig(str(OUTPUT_DIR / "frequency_histogram.png"))
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

n, bins, patches = plt.hist(
    df['recency'],
    bins=30,
    edgecolor='black',   # black border
    rwidth=0.9           # small gaps between bars
)

# hatch pattern on all bars
for patch in patches:
    patch.set_hatch('//')
    patch.set_facecolor('white')

plt.xlabel('Recency')
plt.ylabel('Number of Customers')
plt.title('Distribution of Recency')

plt.show()


In [ ]:
plt.figure(figsize=(6,4))

n, bins, patches = plt.hist(
    df['recency'],
    bins=30,
    edgecolor='black',
    rwidth=0.9
)

for patch in patches:
    patch.set_hatch('//')
    patch.set_facecolor('white')

plt.xlabel('Recency')
plt.ylabel('Number of Customers')
plt.title('Distribution of Recency')

# Save as PNG
plt.savefig(
    'recency_distribution.png',
    dpi=600,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# FREQUENCY
plt.figure(figsize=(6,4))

plt.hist(
    df['frequency'],
    bins=30,
    color='lightgray',
    edgecolor='black',
    rwidth=0.9
)

plt.xlabel('Frequency')
plt.ylabel('Number of Customers')
plt.title('Distribution of Frequency')

plt.tight_layout()

plt.savefig(
    str(OUTPUT_DIR / "frequency_histogram.png"),
    dpi=600,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# MONETARY
plt.figure(figsize=(6,4))
plt.hist(df['monetary'], bins=30)
plt.xlabel('Monetary')
plt.ylabel('Number of Customers')
plt.title('Distribution of Monetary')
plt.tight_layout()

plt.savefig(str(OUTPUT_DIR / "monetary_histogram.png"))
plt.show()

In [ ]:
# MONETARY
plt.figure(figsize=(6,4))

n, bins, patches = plt.hist(
    df['monetary'],
    bins=30,
    edgecolor='black',
    rwidth=0.9
)

# Add hatch pattern
for patch in patches:
    patch.set_hatch('//')
    patch.set_facecolor('white')

plt.xlabel('Monetary')
plt.ylabel('Number of Customers')
plt.title('Distribution of Monetary')

plt.tight_layout()

plt.savefig(
    str(OUTPUT_DIR / "monetary_histogram.png"),
    dpi=600,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# FREQUENCY
plt.figure(figsize=(6,4))

plt.hist(
    df['frequency'],
    bins=30,
    color='lightgray',
    edgecolor='black',
    rwidth=0.9
)

plt.xlabel('Frequency')
plt.ylabel('Number of Customers')
plt.title('Distribution of Frequency')

plt.tight_layout()

plt.savefig(
    str(OUTPUT_DIR / "frequency_histogram.png"),
    dpi=600,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# FREQUENCY
plt.figure(figsize=(6,4))

n, bins, patches = plt.hist(
    df['frequency'],
    bins=30,
    edgecolor='black',
    rwidth=0.9
)

# Add hatch pattern
for patch in patches:
    patch.set_hatch('//')
    patch.set_facecolor('white')

plt.xlabel('Frequency')
plt.ylabel('Number of Customers')
plt.title('Distribution of Frequency')

plt.tight_layout()

plt.savefig(
    str(OUTPUT_DIR / "frequency_histogram.png"),
    dpi=600,
    bbox_inches='tight'
)

plt.show()

In [ ]:
import pandas as pd

# load file
df = pd.read_csv(str(OUTPUT_DIR / "final_selected_features.csv"))

# check first few rows
print(df.head())

# check shape
print("Shape:", df.shape)

# check data types
print("\nData Types:")
print(df.dtypes)

# check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# check duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())

# check duplicate customer ids
print("\nDuplicate Customer IDs:", df['customer_id'].duplicated().sum())

# check zero values in important columns
columns_to_check = ['recency', 'adaptive_recency', 'frequency', 'monetary']

for col in columns_to_check:
    print(f"\nZero values in {col}: {(df[col] == 0).sum()}")

# check negative values
for col in columns_to_check:
    print(f"Negative values in {col}: {(df[col] < 0).sum()}")

# summary statistics
print("\nSummary Statistics:")
print(df[columns_to_check].describe())

# check unique values count
print("\nUnique Values:")
for col in columns_to_check:
    print(f"{col}: {df[col].nunique()}")

In [ ]:
features = ['recency', 'frequency', 'monetary', 'adaptive_recency']

# count exact duplicate rows
duplicate_count = df.duplicated(subset=features).sum()

print("Number of duplicate rows:", duplicate_count)

In [ ]:
# check skewness
print(df[['recency', 'adaptive_recency', 'frequency', 'monetary']].skew())

# check extreme values using percentiles
print(df[['recency', 'adaptive_recency', 'frequency', 'monetary']].quantile([0.90, 0.95, 0.99]))

## K-Means experiments

### K-Means++ on RFM

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# features
features = ['recency', 'frequency', 'monetary']

# keep only needed columns
df_kmeans = df[features].dropna().copy()

# scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_kmeans)

# elbow method
inertia_values = []
K_range = range(2, 11)   # 2 se 10 clusters check kar rahe hain

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

# plot elbow curve
plt.figure(figsize=(8,5))
plt.plot(K_range, inertia_values, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia / WCSS')
plt.title('Elbow Method for KMeans')
plt.show()

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
import numpy as np
features = ['recency', 'frequency', 'monetary']

# keep only needed columns and drop missing values
df_kmeans = df[features].dropna().copy()

print("Shape of data used for clustering:", df_kmeans.shape)

# -----------------------------
# 2. Scale the data
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_kmeans)

# -----------------------------
# 3. Apply KMeans++
# -----------------------------
kmeans_pp = KMeans(
    n_clusters=4,
    init='k-means++',   # explicit KMeans++
    n_init=10,
    random_state=42
) 



df_kmeans['cluster_kmeans_pp'] = kmeans_pp.fit_predict(X_scaled)

# -----------------------------
# 4. Cluster counts
# -----------------------------
print("\nCluster Counts:")
print(df_kmeans['cluster_kmeans_pp'].value_counts().sort_index())

# -----------------------------
# 5. Cluster summary
# -----------------------------
cluster_summary = df_kmeans.groupby('cluster_kmeans_pp')[features].mean()

print("\nCluster Summary:")
print(cluster_summary)

# Preserve the chosen RFM-only labels before later experiments overwrite variables.
rfm_labels = pd.Series(df_kmeans["cluster_kmeans_pp"].to_numpy(),
                       index=df.loc[df_kmeans.index, "customer_id"].to_numpy(),
                       name="cluster_kmeans")
if not rfm_labels.index.is_unique:
    raise ValueError("customer_id must be unique before exporting cluster labels.")


In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.utils import resample

# labels from your clustering
labels = df_kmeans['cluster_kmeans_pp'].values

# -----------------------------
# Silhouette Score on sample only
# -----------------------------
X_sample, labels_sample = resample(
    X_scaled,
    labels,
    n_samples=min(20000, len(labels)),
    random_state=42,
    stratify=labels
)

silhouette = silhouette_score(X_sample, labels_sample)

# -----------------------------
# CH and DB on full data
# -----------------------------
calinski = calinski_harabasz_score(X_scaled, labels)
davies = davies_bouldin_score(X_scaled, labels)

print("KMeans++ Evaluation Metrics")
print("Silhouette Score (sample = 20,000):", silhouette)
print("Calinski-Harabasz Score (full data):", calinski)
print("Davies-Bouldin Score (full data):", davies)

### K-Means++ on RFM + adaptive recency

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# features
features = ['recency', 'adaptive_recency', 'frequency', 'monetary']

# keep only needed columns
df_kmeans = df[features].dropna().copy()

# scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_kmeans)

# elbow method
inertia_values = []
K_range = range(2, 11)   # 2 se 10 clusters check kar rahe hain

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

# plot elbow curve
plt.figure(figsize=(8,5))
plt.plot(K_range, inertia_values, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia / WCSS')
plt.title('Elbow Method for KMeans')
plt.show()

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
import numpy as np
features = ['recency', 'frequency','adaptive_recency' ,'monetary']

# keep only needed columns and drop missing values
df_kmeans = df[features].dropna().copy()

print("Shape of data used for clustering:", df_kmeans.shape)

# -----------------------------
# 2. Scale the data
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_kmeans)

# -----------------------------
# 3. Apply KMeans++
# -----------------------------
kmeans_pp = KMeans(
    n_clusters=4,
    init='k-means++',   # explicit KMeans++
    n_init=10,
    random_state=42
) 



df_kmeans['cluster_kmeans_pp'] = kmeans_pp.fit_predict(X_scaled)

# -----------------------------
# 4. Cluster counts
# -----------------------------
print("\nCluster Counts:")
print(df_kmeans['cluster_kmeans_pp'].value_counts().sort_index())

# -----------------------------
# 5. Cluster summary
# -----------------------------
cluster_summary = df_kmeans.groupby('cluster_kmeans_pp')[features].mean()

print("\nCluster Summary:")
print(cluster_summary)


In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.utils import resample

# labels from your clustering
labels = df_kmeans['cluster_kmeans_pp'].values

# -----------------------------
# Silhouette Score on sample only
# -----------------------------
X_sample, labels_sample = resample(
    X_scaled,
    labels,
    n_samples=min(20000, len(labels)),
    random_state=42,
    stratify=labels
)

silhouette = silhouette_score(X_sample, labels_sample)

# -----------------------------
# CH and DB on full data
# -----------------------------
calinski = calinski_harabasz_score(X_scaled, labels)
davies = davies_bouldin_score(X_scaled, labels)

print("KMeans++ Evaluation Metrics")
print("Silhouette Score (sample = 20,000):", silhouette)
print("Calinski-Harabasz Score (full data):", calinski)
print("Davies-Bouldin Score (full data):", davies)

In [ ]:
# Join the saved RFM-only labels by customer ID, never by row position.
df_final = pd.read_csv(str(OUTPUT_DIR / "final_selected_features.csv"))
df_merged = df_final.merge(
    rfm_labels.rename_axis("customer_id").reset_index(),
    on="customer_id", how="inner", validate="one_to_one"
)
print("Customers with valid RFM labels:", len(df_merged))
print("Excluded rows without RFM labels:", len(df_final) - len(df_merged))
df_merged.to_csv(str(OUTPUT_DIR / "final_with_labels.csv"), index=False)


## FCA-style cluster explanations

Rules are computed from discretized RFM attributes. The cluster-level concept lattice uses the `concepts` package and rules selected from the current run.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(str(OUTPUT_DIR / "final_with_labels.csv"))

# required columns
required = ["customer_id", "recency", "frequency", "monetary", "cluster_kmeans"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# -----------------------------
# 1. Make RFM scores if not already present
# -----------------------------
def quintile_score(s, ascending=True):
    ranks = s.rank(method="first", na_option="keep")
    labels = [1, 2, 3, 4, 5] if ascending else [5, 4, 3, 2, 1]
    return pd.qcut(ranks, q=5, labels=labels, duplicates="drop").astype("Int8")

if "R_score" not in df.columns:
    df["R_score"] = quintile_score(df["recency"], ascending=False)

if "F_score" not in df.columns:
    df["F_score"] = quintile_score(df["frequency"], ascending=True)

if "M_score" not in df.columns:
    df["M_score"] = quintile_score(df["monetary"], ascending=True)

# -----------------------------
# 2. Create FCA attributes
# -----------------------------
df["cluster_attr"] = "Cluster_" + df["cluster_kmeans"].astype(str)
df["R_attr"] = "R" + df["R_score"].astype(str)
df["F_attr"] = "F" + df["F_score"].astype(str)
df["M_attr"] = "M" + df["M_score"].astype(str)

# -----------------------------
# 3. Make binary FCA context
# rows = customers
# columns = Cluster + RFM attributes
# -----------------------------
attr_cols = ["cluster_attr", "R_attr", "F_attr", "M_attr"]

all_attrs = sorted(
    pd.unique(df[attr_cols].values.ravel())
)

context = pd.DataFrame(False, index=df["customer_id"], columns=all_attrs)

for col in attr_cols:
    temp = pd.crosstab(df["customer_id"], df[col]).astype(bool)
    context.update(temp)

context = context.astype(bool)

print("FCA context shape:", context.shape)
print(context.head())

# -----------------------------
# 4. FCA-style implication:
# Cluster_X -> common R/F/M attributes
# -----------------------------
results = []

cluster_attrs = sorted(df["cluster_attr"].unique())

for c in cluster_attrs:
    cluster_customers = context[context[c] == True]
    cluster_size = len(cluster_customers)

    for attr in context.columns:
        if attr.startswith("Cluster_"):
            continue

        support_count = cluster_customers[attr].sum()
        confidence = support_count / cluster_size if cluster_size > 0 else 0

        results.append({
            "cluster": c,
            "attribute": attr,
            "cluster_size": cluster_size,
            "support_count": int(support_count),
            "confidence": confidence
        })

rules = pd.DataFrame(results)

# keep strong explanations only
strong_rules = rules[rules["confidence"] >= 0.50].copy()
strong_rules = strong_rules.sort_values(
    ["cluster", "confidence", "support_count"],
    ascending=[True, False, False]
)

print("Strong FCA explanations:")
print(strong_rules)

strong_rules.to_csv(str(OUTPUT_DIR / "fca_cluster_explanations.csv"), index=False)

# -----------------------------
# 5. Create readable explanation per cluster
# -----------------------------
explanations = []

for c in cluster_attrs:
    temp = strong_rules[strong_rules["cluster"] == c]

    attrs = temp["attribute"].tolist()
    confs = temp["confidence"].round(2).tolist()

    explanation = ", ".join(
        [f"{a} ({conf*100:.0f}%)" for a, conf in zip(attrs, confs)]
    )

    explanations.append({
        "cluster": c,
        "explanation": explanation
    })

explanation_df = pd.DataFrame(explanations)

print(explanation_df)

explanation_df.to_csv(
    str(OUTPUT_DIR / "readable_fca_cluster_explanations.csv"),
    index=False
)

In [ ]:
# -----------------------------
# Show ALL FCA-style rules
# Cluster_X -> attribute
# -----------------------------

all_rules = rules.copy()

all_rules = all_rules.sort_values(
    ["cluster", "confidence", "support_count"],
    ascending=[True, False, False]
)

print("Total rules:", len(all_rules))
print(all_rules.head(100))

# full file save
all_rules.to_csv(str(OUTPUT_DIR / "all_fca_rules.csv"), index=False)

print("✅ All rules saved at: /kaggle/working/all_fca_rules.csv")

In [ ]:
rules_filtered = rules[rules["confidence"] >= 0.5].copy()

In [ ]:
def select_best_rules(cluster_df):
    selected = []
    
    for prefix in ["R", "F", "M"]:
        temp = cluster_df[cluster_df["attribute"].str.startswith(prefix)]
        
        if not temp.empty:
            best = temp.sort_values("confidence", ascending=False).iloc[0]
            selected.append(best)
    
    return pd.DataFrame(selected)

In [ ]:
final_rules = (
    rules_filtered
    .groupby("cluster", group_keys=False)
    .apply(select_best_rules)
)

final_rules = final_rules.sort_values(["cluster", "attribute"])

print(final_rules)

In [ ]:
rules_filtered = rules[rules["confidence"] >= 0.3].copy()

In [ ]:
def select_best_rules(cluster_df):
    selected = []
    
    for prefix in ["R", "F", "M"]:
        temp = cluster_df[cluster_df["attribute"].str.startswith(prefix)]
        
        if not temp.empty:
            best = temp.sort_values("confidence", ascending=False).iloc[0]
            selected.append(best)
    
    return pd.DataFrame(selected)

In [ ]:
final_rules = (
    rules_filtered
    .groupby("cluster", group_keys=False)
    .apply(select_best_rules)
)

final_rules = final_rules.sort_values(["cluster", "attribute"])

print(final_rules)

In [ ]:
rules_filtered = rules[rules["confidence"] >= 0.2].copy()

In [ ]:
def select_best_rules(cluster_df):
    selected = []
    
    for prefix in ["R", "F", "M"]:
        temp = cluster_df[cluster_df["attribute"].str.startswith(prefix)]
        
        if not temp.empty:
            best = temp.sort_values("confidence", ascending=False).iloc[0]
            selected.append(best)
    
    return pd.DataFrame(selected)

In [ ]:
final_rules = (
    rules_filtered
    .groupby("cluster", group_keys=False)
    .apply(select_best_rules)
)

final_rules = final_rules.sort_values(["cluster", "attribute"])

print(final_rules)

In [ ]:
from concepts import Context
from graphviz import Digraph

# Build the lattice from the current run's selected rules.
rules_df = final_rules[["cluster", "attribute"]].drop_duplicates().copy()
if rules_df.empty:
    raise ValueError("No rules passed the selected confidence threshold.")
print(rules_df)


In [ ]:
# -----------------------------
# 2. Create FCA binary context
# -----------------------------
objects = sorted(rules_df["cluster"].unique())
attributes = sorted(rules_df["attribute"].unique())

binary_context = pd.DataFrame(False, index=objects, columns=attributes)

for _, row in rules_df.iterrows():
    binary_context.loc[row["cluster"], row["attribute"]] = True

print(binary_context)

binary_context.to_csv(str(OUTPUT_DIR / "cluster_level_fca_context.csv"))

In [ ]:
# -----------------------------
# 3. Create Formal Context
# -----------------------------
context = Context(
    objects,
    attributes,
    binary_context.values.tolist()
)

print(context)

In [ ]:
# -----------------------------
# 4. Generate concept lattice
# -----------------------------
lattice = context.lattice

print("Number of concepts:", len(lattice))

for i, concept in enumerate(lattice):
    print(f"\nConcept {i}")
    print("Extent / Objects:", list(concept.extent))
    print("Intent / Attributes:", list(concept.intent))

In [ ]:
# -----------------------------
# 5. Draw lattice using concept.upper_neighbors / lower_neighbors
# -----------------------------
from graphviz import Digraph

dot = Digraph(comment="Cluster Level FCA Concept Lattice")
dot.attr(rankdir="BT")

concept_ids = {}

# create nodes
for i, concept in enumerate(lattice):
    extent = ", ".join(concept.extent) if concept.extent else "∅"
    intent = ", ".join(concept.intent) if concept.intent else "∅"
    
    label = f"Objects: {extent}\nAttributes: {intent}"
    node_id = f"C{i}"
    
    concept_ids[concept] = node_id
    dot.node(node_id, label=label, shape="box")

# create edges
for concept in lattice:
    for upper in concept.upper_neighbors:
        dot.edge(concept_ids[concept], concept_ids[upper])

# save diagram
output_path = str(OUTPUT_DIR / "cluster_level_fca_lattice")

dot.render(output_path, format="png", cleanup=True)
dot.render(output_path, format="pdf", cleanup=True)

print("✅ PNG saved at:", output_path + ".png")
print("✅ PDF saved at:", output_path + ".pdf")

## Additional RFM + adaptive-recency experiment

In [ ]:
df = pd.read_csv(INPUT_CSV).replace([np.inf, -np.inf], np.nan)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
import numpy as np
features = ['recency', 'adaptive_recency', 'frequency', 'monetary']

# keep only needed columns and drop missing values
df_kmeans = df[features].dropna().copy()

print("Shape of data used for clustering:", df_kmeans.shape)

# -----------------------------
# 2. Scale the data
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_kmeans)

# -----------------------------
# 3. Apply KMeans++
# -----------------------------
kmeans_pp = KMeans(
    n_clusters=4,
    init='k-means++',   # explicit KMeans++
    n_init=10,
    random_state=42
) 



df_kmeans['cluster_kmeans_pp'] = kmeans_pp.fit_predict(X_scaled)

# -----------------------------
# 4. Cluster counts
# -----------------------------
print("\nCluster Counts:")
print(df_kmeans['cluster_kmeans_pp'].value_counts().sort_index())

# -----------------------------
# 5. Cluster summary
# -----------------------------
cluster_summary = df_kmeans.groupby('cluster_kmeans_pp')[features].mean()

print("\nCluster Summary:")
print(cluster_summary)


In [ ]:
from sklearn.metrics import silhouette_score
import numpy as np

# sample size
sample_size = min(50000, X_scaled.shape[0])

# random sample
np.random.seed(42)
sample_idx = np.random.choice(X_scaled.shape[0], sample_size, replace=False)

# sample data and labels
X_sample = X_scaled[sample_idx]
labels_sample = df_kmeans['cluster_kmeans_pp'].to_numpy()[sample_idx]

# silhouette score on sample
sil_score = silhouette_score(X_sample, labels_sample)

print("Sample Silhouette Score:", sil_score)

In [ ]:
ch_score = calinski_harabasz_score(X_scaled, df_kmeans['cluster_kmeans_pp'])
db_score = davies_bouldin_score(X_scaled, df_kmeans['cluster_kmeans_pp'])

print("\nCalinski-Harabasz Score:", ch_score)
print("Davies-Bouldin Score:", db_score)


## BIRCH experiments

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import Birch
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
import numpy as np

# -----------------------------
# 1. Select features
# -----------------------------
features = ['recency', 'adaptive_recency', 'frequency', 'monetary']

# keep only needed columns and drop missing values
df_birch = df[features].dropna().copy()

print("Shape of data used for clustering:", df_birch.shape)

# -----------------------------
# 2. Scale the data
# -----------------------------
scaler = StandardScaler()
X_scaled_birch = scaler.fit_transform(df_birch)

# -----------------------------
# 3. Apply BIRCH
# -----------------------------
birch_model = Birch(
    n_clusters=4,      # final number of clusters
    threshold=0.5,     # can tune later
    branching_factor=50
)

df_birch['cluster_birch'] = birch_model.fit_predict(X_scaled_birch)

# -----------------------------
# 4. Cluster counts
# -----------------------------
print("\nCluster Counts:")
print(df_birch['cluster_birch'].value_counts().sort_index())

# -----------------------------
# 5. Cluster summary
# -----------------------------
cluster_summary = df_birch.groupby('cluster_birch')[features].mean()

print("\nCluster Summary:")
print(cluster_summary)

# -----------------------------
# 6. Evaluation metrics
# -----------------------------
ch_score = calinski_harabasz_score(X_scaled_birch, df_birch['cluster_birch'])
db_score = davies_bouldin_score(X_scaled_birch, df_birch['cluster_birch'])

print("\nCalinski-Harabasz Score:", ch_score)
print("Davies-Bouldin Score:", db_score)

# -----------------------------
# 7. Sample-based silhouette score
# -----------------------------
sample_size = min(50000, X_scaled_birch.shape[0])

np.random.seed(42)
sample_idx = np.random.choice(X_scaled_birch.shape[0], sample_size, replace=False)

X_sample = X_scaled_birch[sample_idx]
labels_sample = df_birch['cluster_birch'].to_numpy()[sample_idx]

sil_score = silhouette_score(X_sample, labels_sample)

print("Sample Silhouette Score:", sil_score)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import Birch
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
import pandas as pd
import numpy as np

# -----------------------------
# 1. Select features
# -----------------------------
features = ['recency', 'adaptive_recency', 'frequency', 'monetary']

df_birch = df[features].dropna().copy()
print("Shape of data used:", df_birch.shape)

# -----------------------------
# 2. Scale data
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_birch)

# -----------------------------
# 3. Sample for silhouette
# -----------------------------
sample_size = min(50000, X_scaled.shape[0])

np.random.seed(42)
sample_idx = np.random.choice(X_scaled.shape[0], sample_size, replace=False)

X_sample = X_scaled[sample_idx]

# -----------------------------
# 4. Try different BIRCH thresholds
# -----------------------------
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0, 1.5]

results = []

for th in thresholds:
    print(f"\nRunning BIRCH with threshold = {th}")

    birch_model = Birch(
        n_clusters=4,
        threshold=th,
        branching_factor=50
    )

    labels = birch_model.fit_predict(X_scaled)

    # avoid error if only 1 cluster is created
    n_unique = len(np.unique(labels))
    print("Unique clusters:", n_unique)

    if n_unique < 2:
        print("Skipped because only one cluster formed.")
        continue

    labels_sample = labels[sample_idx]

    ch = calinski_harabasz_score(X_scaled, labels)
    db = davies_bouldin_score(X_scaled, labels)
    sil = silhouette_score(X_sample, labels_sample)

    results.append({
        'threshold': th,
        'n_clusters': n_unique,
        'silhouette': sil,
        'calinski_harabasz': ch,
        'davies_bouldin': db
    })

# -----------------------------
# 5. Results comparison
# -----------------------------
if not results:
    raise ValueError('No valid BIRCH configuration produced evaluable clusters.')
results_df = pd.DataFrame(results)

print("\nBIRCH Threshold Comparison:")
print(results_df.sort_values(by='silhouette', ascending=False))

# -----------------------------
# 6. Pick best threshold
# -----------------------------
best_threshold = results_df.sort_values(
    by=['silhouette', 'davies_bouldin'],
    ascending=[False, True]
).iloc[0]['threshold']

print("\nBest Threshold Selected:", best_threshold)

# -----------------------------
# 7. Final BIRCH with best threshold
# -----------------------------
final_birch = Birch(
    n_clusters=4,
    threshold=best_threshold,
    branching_factor=50
)

df_birch['cluster_birch'] = final_birch.fit_predict(X_scaled)

# -----------------------------
# 8. Cluster counts
# -----------------------------
print("\nFinal Cluster Counts:")
print(df_birch['cluster_birch'].value_counts().sort_index())

# -----------------------------
# 9. Cluster summary
# -----------------------------
cluster_summary = df_birch.groupby('cluster_birch')[features].mean()

print("\nFinal Cluster Summary:")
print(cluster_summary)

# -----------------------------
# 10. Final evaluation
# -----------------------------
final_labels = df_birch['cluster_birch'].to_numpy()
final_labels_sample = final_labels[sample_idx]

ch_score = calinski_harabasz_score(X_scaled, final_labels)
db_score = davies_bouldin_score(X_scaled, final_labels)
sil_score = silhouette_score(X_sample, final_labels_sample)

print("\nFinal Calinski-Harabasz Score:", ch_score)
print("Final Davies-Bouldin Score:", db_score)
print("Final Sample Silhouette Score:", sil_score)

# -----------------------------
# 11. Save file
# -----------------------------
df_birch.to_csv(str(OUTPUT_DIR / "birch_clustered_file.csv"), index=False)

print("\nSaved: /kaggle/working/birch_clustered_file.csv")

## Agglomerative clustering

This comparison uses a sample because hierarchical clustering is memory intensive.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# =========================================================
# 1) LOAD FILE
# =========================================================
df = pd.read_csv(INPUT_CSV)

# =========================================================
# 2) SELECT FEATURES
# =========================================================
features = [
    "recency",
    "adaptive_recency",
    "frequency",
    "monetary"
]

# =========================================================
# 3) CLEAN DATA
# =========================================================
X = df[features].copy()

print("Initial Shape:", X.shape)

print("\nNull values:")
print(X.isnull().sum())

# fill missing
X = X.fillna(0)

# =========================================================
# 4) SAMPLE DATA
# Agglomerative full data is too heavy
# =========================================================
sample_size = min(15000, len(df))

df_sample = df.sample(n=sample_size, random_state=42).copy()
X_sample = df_sample[features].copy().fillna(0)

print("\nSample Shape:", X_sample.shape)

# =========================================================
# 5) SCALE SAMPLE
# =========================================================
scaler = StandardScaler()
X_sample_scaled = scaler.fit_transform(X_sample)

# =========================================================
# 6) APPLY AGGLOMERATIVE
# =========================================================
agglo_model = AgglomerativeClustering(
    n_clusters=4,
    metric='euclidean',
    linkage='ward'
)

df_sample["cluster_agglo"] = agglo_model.fit_predict(X_sample_scaled)

# =========================================================
# 7) CLUSTER COUNTS
# =========================================================
print("\nCluster Counts:")
print(df_sample["cluster_agglo"].value_counts().sort_index())

# =========================================================
# 8) CLUSTER SUMMARY
# =========================================================
cluster_summary = df_sample.groupby("cluster_agglo")[features].mean()

print("\nCluster Summary:")
print(cluster_summary)

# =========================================================
# 9) EVALUATION METRICS
# =========================================================
labels = df_sample["cluster_agglo"]

sil_score = silhouette_score(X_sample_scaled, labels)
ch_score = calinski_harabasz_score(X_sample_scaled, labels)
db_score = davies_bouldin_score(X_sample_scaled, labels)

print("\nEvaluation Metrics:")
print("Silhouette Score:", sil_score)
print("Calinski-Harabasz Score:", ch_score)
print("Davies-Bouldin Score:", db_score)

# =========================================================
# 10) SAVE RESULTS
# =========================================================
df_sample.to_csv(str(OUTPUT_DIR / "agglomerative_clusters.csv"), index=False)
cluster_summary.to_csv(str(OUTPUT_DIR / "agglomerative_cluster_summary.csv"))

print("\nFiles Saved:")
print(str(OUTPUT_DIR / "agglomerative_clusters.csv"))
print(str(OUTPUT_DIR / "agglomerative_cluster_summary.csv"))